# Bluestock Fintech — Mutual Fund Analytics Capstone
## Day 6: Advanced Analytics + Risk Metrics

Covers Historical VaR/CVaR, Rolling Sharpe Ratio, Investor Cohort Analysis,
SIP Continuation/At-Risk Flagging, Fund Recommendation Logic, and Sector Concentration (HHI).


In [ ]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

def find_data_dir():
    kaggle_matches = glob.glob("/kaggle/input/*/01_fund_master.csv")
    if kaggle_matches:
        return os.path.dirname(kaggle_matches[0])
    if os.path.exists("01_fund_master.csv"):
        return "."
    raise FileNotFoundError("Could not find 01_fund_master.csv — check your Kaggle input dataset.")

DATA_DIR = find_data_dir()
print(f"Reading CSVs from: {DATA_DIR}")

In [ ]:
fund_master  = pd.read_csv(f"{DATA_DIR}/01_fund_master.csv")
nav_history  = pd.read_csv(f"{DATA_DIR}/02_nav_history.csv", parse_dates=["date"])
performance  = pd.read_csv(f"{DATA_DIR}/07_scheme_performance.csv")
transactions = pd.read_csv(f"{DATA_DIR}/08_investor_transactions.csv", parse_dates=["transaction_date"])
holdings     = pd.read_csv(f"{DATA_DIR}/09_portfolio_holdings.csv")

# Sort NAV history and compute daily returns (needed for VaR, rolling Sharpe)
nav_history = nav_history.sort_values(["amfi_code", "date"]).reset_index(drop=True)
nav_history["daily_return"] = nav_history.groupby("amfi_code")["nav"].pct_change()

print("All datasets loaded.")
print(f"Funds: {len(fund_master)} | NAV rows: {len(nav_history)} | Transactions: {len(transactions)}")

## Task 1 — Historical VaR (95%) & CVaR per Fund

VaR = 5th percentile of the daily return distribution (the loss level not expected to be
exceeded on 95% of days). CVaR (Conditional VaR / Expected Shortfall) = the average of all
returns that fall below the VaR threshold — i.e. "if things go bad, how bad on average."


In [ ]:
def compute_var_cvar(returns, confidence=0.95):
    """Historical VaR and CVaR at given confidence level. Returns are daily fractional returns."""
    returns = returns.dropna()
    if len(returns) < 30:
        return np.nan, np.nan
    var_threshold = np.percentile(returns, (1 - confidence) * 100)
    cvar = returns[returns <= var_threshold].mean()
    return var_threshold, cvar

var_cvar_records = []
for code_, grp in nav_history.groupby("amfi_code"):
    var_95, cvar_95 = compute_var_cvar(grp["daily_return"], confidence=0.95)
    var_cvar_records.append({
        "amfi_code": code_,
        "var_95_pct": var_95 * 100 if pd.notna(var_95) else np.nan,
        "cvar_95_pct": cvar_95 * 100 if pd.notna(cvar_95) else np.nan,
    })

var_cvar_df = pd.DataFrame(var_cvar_records)
var_cvar_df = var_cvar_df.merge(fund_master[["amfi_code", "scheme_name", "category", "risk_category"]],
                                  on="amfi_code", how="left")
var_cvar_df = var_cvar_df.sort_values("var_95_pct")

var_cvar_df.to_csv("var_cvar_report.csv", index=False)
print(f"Saved var_cvar_report.csv — {len(var_cvar_df)} funds")
var_cvar_df.head(10)

In [ ]:
# Visualize: which funds carry the highest tail risk
top_risk = var_cvar_df.nsmallest(10, "var_95_pct")

plt.figure(figsize=(12, 6))
plt.barh(top_risk["scheme_name"], top_risk["var_95_pct"], color="indianred", label="VaR (95%)")
plt.barh(top_risk["scheme_name"], top_risk["cvar_95_pct"], color="darkred", alpha=0.6, label="CVaR (95%)")
plt.xlabel("Daily Return (%)")
plt.title("Top 10 Funds by Tail Risk — VaR vs CVaR (95% confidence)", fontsize=13, fontweight="bold")
plt.legend()
plt.tight_layout()
plt.savefig("chart_var_cvar.png", dpi=120)
plt.show()

## Task 2 — Rolling 90-day Sharpe Ratio (5 Funds)

Rolling Sharpe shows how a fund's risk-adjusted return evolves over time, rather than
a single static number. Uses Rf = 6.5% (RBI repo rate proxy), annualised with sqrt(252).


In [ ]:
RF_ANNUAL = 0.065
RF_DAILY = RF_ANNUAL / 252

# Pick 5 funds for readability — first 5 alphabetically by scheme name
sample_funds = fund_master.sort_values("scheme_name")["amfi_code"].head(5).tolist()

plt.figure(figsize=(14, 7))
for code_ in sample_funds:
    fund_data = nav_history[nav_history["amfi_code"] == code_].copy()
    fund_data = fund_data.set_index("date")
    excess_return = fund_data["daily_return"] - RF_DAILY
    rolling_sharpe = (
        excess_return.rolling(90).mean() / excess_return.rolling(90).std()
    ) * np.sqrt(252)
    fund_name = fund_master.loc[fund_master["amfi_code"] == code_, "scheme_name"].values[0]
    plt.plot(rolling_sharpe.index, rolling_sharpe.values, label=fund_name, linewidth=1.2)

plt.axhline(0, color="black", linewidth=0.8, linestyle="--")
plt.title("Rolling 90-Day Sharpe Ratio — 5 Sample Funds", fontsize=13, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Rolling Sharpe Ratio (annualised)")
plt.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("rolling_sharpe_chart.png", dpi=120)
plt.show()

## Task 3 — Investor Cohort Analysis

Groups investors by the year of their first transaction, then compares average SIP
amount, total invested, and fund category preference across cohorts.


In [ ]:
first_tx = transactions.groupby("investor_id")["transaction_date"].min().reset_index()
first_tx["cohort_year"] = first_tx["transaction_date"].dt.year
first_tx = first_tx.rename(columns={"transaction_date": "first_transaction_date"})

tx_with_cohort = transactions.merge(first_tx[["investor_id", "cohort_year"]], on="investor_id")

sip_tx = tx_with_cohort[tx_with_cohort["transaction_type"] == "SIP"]
cohort_summary = sip_tx.groupby("cohort_year").agg(
    avg_sip_amount=("amount_inr", "mean"),
    total_invested=("amount_inr", "sum"),
    num_investors=("investor_id", "nunique"),
    num_transactions=("investor_id", "count"),
).reset_index()

# Fund category preference per cohort — top category by transaction count
tx_with_fund = tx_with_cohort.merge(fund_master[["amfi_code", "category"]], on="amfi_code")
top_category = (
    tx_with_fund.groupby(["cohort_year", "category"]).size()
    .reset_index(name="count")
    .sort_values(["cohort_year", "count"], ascending=[True, False])
    .groupby("cohort_year").first()[["category"]]
    .rename(columns={"category": "top_category"})
)

cohort_summary = cohort_summary.merge(top_category, on="cohort_year", how="left")
cohort_summary.to_csv("cohort_analysis.csv", index=False)
print(f"Saved cohort_analysis.csv — {len(cohort_summary)} cohorts")
cohort_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].bar(cohort_summary["cohort_year"].astype(str), cohort_summary["avg_sip_amount"], color="steelblue")
axes[0].set_title("Average SIP Amount by Cohort Year", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Avg SIP Amount (Rs.)")
axes[0].set_xlabel("Cohort Year (First Transaction)")

axes[1].bar(cohort_summary["cohort_year"].astype(str), cohort_summary["num_investors"], color="seagreen")
axes[1].set_title("Number of Investors by Cohort Year", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Investor Count")
axes[1].set_xlabel("Cohort Year (First Transaction)")

plt.tight_layout()
plt.savefig("chart_cohort_analysis.png", dpi=120)
plt.show()

## Task 4 — SIP Continuation Analysis

For investors with 6+ SIP transactions, computes the average gap between consecutive
SIPs. Investors with an average gap exceeding 35 days are flagged as **at-risk**
(i.e. likely to have lapsed or be inconsistent with their SIP discipline).


In [ ]:
sip_only = transactions[transactions["transaction_type"] == "SIP"].sort_values(["investor_id", "transaction_date"])

sip_counts = sip_only.groupby("investor_id").size()
eligible_investors = sip_counts[sip_counts >= 6].index

sip_eligible = sip_only[sip_only["investor_id"].isin(eligible_investors)].copy()
sip_eligible["gap_days"] = sip_eligible.groupby("investor_id")["transaction_date"].diff().dt.days

continuity = sip_eligible.groupby("investor_id").agg(
    num_sips=("transaction_date", "count"),
    avg_gap_days=("gap_days", "mean"),
    max_gap_days=("gap_days", "max"),
).reset_index()

continuity["at_risk"] = continuity["avg_gap_days"] > 35

continuity.to_csv("sip_continuity.csv", index=False)
print(f"Saved sip_continuity.csv — {len(continuity)} investors with 6+ SIPs")
print(f"At-risk investors (avg gap > 35 days): {continuity['at_risk'].sum()} "
      f"({continuity['at_risk'].mean()*100:.1f}% of eligible investors)")
continuity.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(continuity["avg_gap_days"], bins=30, color="steelblue", kde=True)
plt.axvline(35, color="red", linestyle="--", linewidth=1.5, label="At-Risk Threshold (35 days)")
plt.title("Distribution of Average SIP Gap Days", fontsize=13, fontweight="bold")
plt.xlabel("Average Gap Between SIP Transactions (days)")
plt.ylabel("Number of Investors")
plt.legend()
plt.tight_layout()
plt.savefig("chart_sip_continuity.png", dpi=120)
plt.show()

## Task 5 — Simple Fund Recommendation Logic

Input: investor risk appetite. Output: top 3 funds by Sharpe ratio within the matching
risk grade. Note: this dataset uses 5 risk grades (Low, Moderate, Moderately High, High,
Very High) rather than just 3 — the recommender maps a simplified Low/Moderate/High
investor input to the closest matching grades.


In [ ]:
def recommend_funds(risk_appetite, top_n=3):
    """
    risk_appetite: 'Low', 'Moderate', or 'High'
    Maps simplified investor appetite to the dataset's 5 SEBI risk grades.
    """
    risk_map = {
        "Low":      ["Low"],
        "Moderate": ["Moderate", "Moderately High"],
        "High":     ["High", "Very High"],
    }

    if risk_appetite not in risk_map:
        raise ValueError("risk_appetite must be one of: Low, Moderate, High")

    matching_grades = risk_map[risk_appetite]
    candidates = performance.drop(columns=["scheme_name", "category"], errors="ignore").merge(
        fund_master[["amfi_code", "scheme_name", "risk_category", "category"]],
        on="amfi_code"
    )
    candidates = candidates[candidates["risk_category"].isin(matching_grades)]
    top_funds = candidates.sort_values("sharpe_ratio", ascending=False).head(top_n)

    return top_funds[["scheme_name", "category", "risk_category", "sharpe_ratio",
                       "return_3yr_pct", "expense_ratio_pct"]]


print("=== Recommendations for LOW risk appetite ===")
print(recommend_funds("Low").to_string(index=False))

print("\n=== Recommendations for MODERATE risk appetite ===")
print(recommend_funds("Moderate").to_string(index=False))

print("\n=== Recommendations for HIGH risk appetite ===")
print(recommend_funds("High").to_string(index=False))

## Task 6 — Sector Concentration Analysis (HHI)

Herfindahl-Hirschman Index = sum of squared sector weights per fund. Higher HHI means
the fund's equity holdings are concentrated in fewer sectors (higher concentration risk);
lower HHI means broader diversification.


In [ ]:
def compute_hhi(group):
    weights_fraction = group["weight_pct"] / 100
    return (weights_fraction ** 2).sum() * 10000  # scaled 0–10000, standard HHI convention

hhi_df = holdings.groupby("amfi_code").apply(compute_hhi, include_groups=False).reset_index()
hhi_df.columns = ["amfi_code", "hhi"]

hhi_df = hhi_df.merge(fund_master[["amfi_code", "scheme_name", "category", "sub_category"]],
                       on="amfi_code", how="left")
hhi_df = hhi_df.sort_values("hhi", ascending=False)

hhi_df.to_csv("sector_hhi.csv", index=False)
print(f"Saved sector_hhi.csv — {len(hhi_df)} funds with holdings data")
hhi_df.head(10)

In [ ]:
plt.figure(figsize=(12, 7))
colors = plt.cm.RdYlGn_r((hhi_df["hhi"] - hhi_df["hhi"].min()) / (hhi_df["hhi"].max() - hhi_df["hhi"].min()))
plt.barh(hhi_df["scheme_name"], hhi_df["hhi"], color=colors)
plt.xlabel("HHI (higher = more concentrated)")
plt.title("Sector Concentration (HHI) by Fund", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sector_hhi_chart.png", dpi=120)
plt.show()

## Task 7 — Advanced Analytics Summary

In [ ]:
highest_var_fund = var_cvar_df.loc[var_cvar_df["var_95_pct"].idxmin(), "scheme_name"]
best_cohort = cohort_summary.loc[cohort_summary["avg_sip_amount"].idxmax(), "cohort_year"]
at_risk_pct = continuity["at_risk"].mean() * 100
most_concentrated_fund = hhi_df.iloc[0]["scheme_name"]
least_concentrated_fund = hhi_df.iloc[-1]["scheme_name"]

print(f"Fund with highest tail risk (lowest VaR)   : {highest_var_fund}")
print(f"Cohort with highest avg SIP amount         : {best_cohort}")
print(f"Share of eligible investors flagged at-risk: {at_risk_pct:.1f}%")
print(f"Most sector-concentrated fund (highest HHI): {most_concentrated_fund}")
print(f"Most diversified fund (lowest HHI)         : {least_concentrated_fund}")

**Key Insights:**

1. Tail risk (VaR/CVaR) varies meaningfully across the fund universe — equity-oriented
   and small/mid-cap schemes show materially worse downside-day outcomes than debt or
   liquid schemes, as expected given their underlying volatility profile.
2. Rolling Sharpe ratios fluctuate over time rather than staying static, underscoring why
   a single point-in-time Sharpe number can be misleading for fund selection.
3. Investor cohorts differ in both ticket size and category preference — newer cohorts
   in this dataset tend to gravitate toward different fund categories than earlier ones,
   suggesting changing investor behaviour or product availability over time.
4. A non-trivial share of investors with established SIP habits (6+ transactions) show
   inconsistent contribution patterns (avg gap > 35 days), representing a churn-risk
   segment that distributors/AMCs could proactively re-engage. Note: in this dataset the
   median gap between SIP transactions is ~60 days rather than a strict monthly cadence,
   so the at-risk share comes out high under the 35-day rule — worth recalibrating the
   threshold (e.g. 45–60 days) if this logic is reused on data with genuinely monthly SIPs.
5. Sector concentration (HHI) is uneven across equity funds — some schemes carry
   meaningfully more concentrated sector exposure than others, which matters for
   investors evaluating diversification within their equity allocation, separate from
   fund-level category labels alone.

*Note: specific fund names and percentages above are computed directly from this run's
output — refer to the printed values for exact figures.*
